# R5-N1: clean sealed evaluation (Colab)

Preflight is the default and reads no test rows or audio. The run uses the eight locked bundles and writes directly to persistent Drive. Protocol D is not measured. Record GR4 completion or an approved exception, clean-only change control, and the reported test-access history before a sealed run. This notebook does not make those decisions for the team.

Sync the current code, tests, both locks, all eight bundles and A-lock evidence, `reports/r1/split_report.json`, and the Drive `data/` directory before starting. Use a GPU runtime for AST. For a disconnect, preserve the whole output folder and consult the recovery cell; do not start a new run or delete evidence.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, json, os, subprocess, sys, tempfile, xml.etree.ElementTree as ET

# Change only if your Drive project is elsewhere. Keep the data and results on Drive.
PROJECT = Path('/content/drive/MyDrive/PAI&AIES Project')
SHARDS = PROJECT / 'data/processed/cohort_v1'
RUN_ID = 'r5_n1_clean_v1'

# Safe defaults. Set these only after the team has resolved the recorded prerequisites.
GR4_VERIFIED = False
GR4_EXCEPTION_APPROVED = False  # Muneeb approved proceeding with GR4 still open
SCOPE_CHANGE_CONTROL_RESOLVED = False
PRIOR_TEST_ACCESS_RECORDED = False
RUN_SEALED = False
RESUME = False
RESUME_REASON = ''
RECOVER_STALE_CLAIM = False  # only after confirming the old worker is stopped
RUN_ANALYSIS = False

if not PROJECT.is_dir():
    raise FileNotFoundError(f'Project folder unavailable: {PROJECT}. Check Drive mount and shortcut.')
if not RUN_ID or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in RUN_ID):
    raise ValueError('RUN_ID may use only letters, digits, hyphens and underscores.')
ROOT = PROJECT.resolve(strict=True)
SHARDS = SHARDS.resolve(strict=True)
OUT = ROOT / 'reports/final/results' / RUN_ID
required = [
    'scripts/r5_n1_sealed.py', 'scripts/r5_n1_clean_analysis.py',
    'src/echoproof/evaluation/r5_loaders.py', 'src/echoproof/evaluation/r5_clean_analysis.py',
    'tests/test_r5_sealed_runner.py', 'tests/test_r5_loaders.py', 'tests/test_r5_clean_analysis.py', 'tests/test_r5_n1_notebook.py',
    'locks/A_lock.json', 'locks/B_lock_gr3.json', 'reports/r1/split_report.json',
    'data/splits/protocol_A_v1.parquet', 'data/splits/protocol_B_v1.parquet',
]
missing = [name for name in required if not (ROOT / name).is_file()]
if missing: raise FileNotFoundError('Missing project files: ' + ', '.join(missing))
if not (SHARDS / 'index.parquet').is_file() or not list(SHARDS.glob('shard_*.tar')):
    raise FileNotFoundError('Shard index or tar files are missing from ' + str(SHARDS))
print('Project:', ROOT, '\nShards:', SHARDS, '\nPersistent output:', OUT)


## Environment

Install the verified Transformers version and any missing dependencies. Restart the runtime and rerun from the first cell if Colab reports that a package imported earlier must be reloaded. The package versions and hardware are bound to a run on execution; recovery requires them to match.


In [ ]:
import importlib.util
packages = {'numpy':'numpy', 'scipy':'scipy', 'sklearn':'scikit-learn', 'pyarrow':'pyarrow',
            'librosa':'librosa', 'soundfile':'soundfile', 'matplotlib':'matplotlib',
            'pytest':'pytest', 'yaml':'PyYAML', 'torch':'torch', 'huggingface_hub':'huggingface-hub'}
missing_packages = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                'transformers==5.22.0', *missing_packages], check=True)
import importlib.metadata as metadata
if metadata.version('transformers') != '5.22.0':
    raise RuntimeError('Transformers version does not match 5.22.0; restart the runtime.')
import torch
if not torch.cuda.is_available(): raise RuntimeError('Select a Colab GPU runtime; both AST bundles require CUDA FP16.')
print('Python:', sys.version.split()[0], 'GPU:', torch.cuda.get_device_name(0))
for package in ['torch', 'transformers', 'numpy', 'scipy', 'scikit-learn', 'librosa', 'soundfile', 'pyarrow', 'matplotlib']:
    print(package, metadata.version(package))
ENV = os.environ.copy()
ENV['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + ENV.get('PYTHONPATH', '')
ENV['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
ENV['HF_HUB_OFFLINE'] = '1'
ENV['TRANSFORMERS_OFFLINE'] = '1'
def run_command(args, *, capture=False):
    # A stable cwd avoids using a stale resolved Drive shortcut as the process cwd.
    result = subprocess.run([sys.executable, *map(str, args)], cwd='/content', env=ENV,
                            text=True, stdout=subprocess.PIPE if capture else None,
                            stderr=subprocess.STDOUT if capture else None)
    if capture: print(result.stdout, end='')
    if result.returncode: raise RuntimeError(f'Command failed with exit code {result.returncode}; inspect output above.')
    return result.stdout if capture else None


## Synthetic tests

These tests use fabricated data. They must pass without skips before the full preflight.


In [ ]:
TESTS_OK = False
PREFLIGHT_OK = False
tests = ['test_r5_sealed_runner.py', 'test_r5_loaders.py', 'test_r5_clean_analysis.py', 'test_r5_n1_notebook.py']
with tempfile.TemporaryDirectory(prefix='r5_n1_tests_') as folder:
    junit = Path(folder) / 'results.xml'
    run_command(['-m', 'pytest', '-q', '-p', 'no:cacheprovider', '--junitxml', junit,
                 *[ROOT / 'tests' / name for name in tests]])
    suites = ET.parse(junit).getroot()
    skipped = sum(int(node.get('skipped', '0')) for node in suites.iter('testsuite'))
    if skipped: raise RuntimeError(f'{skipped} tests skipped; resolve before the run.')
TESTS_OK = True
print('Synthetic tests passed without skips.')


## Full preflight and bundle loading

The full plan verifies both locks, A evidence, eight bundles, split file presence and hashes, GPU availability and shards. `--load-check` loads each bundle without scoring audio. It does not verify every shard against an independently locked hash; the run records shard hashes. The B split row digest is checked only after logged sealed access.


In [ ]:
PREFLIGHT_OK = False
if not TESTS_OK: raise RuntimeError('Run the synthetic tests first.')
def source_snapshot():
    files = list((ROOT / 'src/echoproof').rglob('*.py'))
    files += [ROOT / 'scripts/r5_n1_sealed.py', ROOT / 'scripts/r5_n1_clean_analysis.py']
    files += [ROOT / 'tests' / name for name in tests]
    return {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(files)}
preflight = run_command([ROOT/'scripts/r5_n1_sealed.py', '--root', ROOT, 'plan',
                         '--shards', SHARDS, '--load-check'], capture=True)
if 'Locks, bundles and split files verified: YES' not in preflight or 'Ready to run on this machine: YES' not in preflight:
    raise RuntimeError('Full preflight did not report readiness; do not open the sealed test.')
PREFLIGHT_SNAPSHOT = source_snapshot()
PREFLIGHT_OK = True
print('Full preflight passed; no test rows or audio were opened.')


## Sealed run ? disabled by default

Before enabling, record the test-access history and clean-only change-control decision; verify GR4 or the approved GR4 exception. Set `GR4_EXCEPTION_APPROVED`, `SCOPE_CHANGE_CONTROL_RESOLVED`, `PRIOR_TEST_ACCESS_RECORDED` and `RUN_SEALED` to `True` in the first code cell, then rerun from the first cell. A new run refuses an existing output folder. It records the access log before opening test rows. Results, journals and the log stay on Drive.

For a genuinely interrupted run, use the same `RUN_ID`, set `RESUME=True` and give `RESUME_REASON`. If `active_run.claim` remains, confirm the old process is stopped before setting `RECOVER_STALE_CLAIM=True`. Recovery rereads the sealed split. A damaged journal, changed identity, integrity failure or ambiguous in-flight recording stops for manual review; do not start a new run to bypass it.


In [ ]:
if not RUN_SEALED:
    print('Sealed run disabled. Preflight only; no test data opened.')
else:
    if not (GR4_VERIFIED or GR4_EXCEPTION_APPROVED) or not all([SCOPE_CHANGE_CONTROL_RESOLVED, PRIOR_TEST_ACCESS_RECORDED]):
        raise RuntimeError('Resolve and record all three prerequisites before the sealed run.')
    if not TESTS_OK or not PREFLIGHT_OK or source_snapshot() != PREFLIGHT_SNAPSHOT:
        raise RuntimeError('Run tests and full preflight again after any code or test changes.')
    if RESUME and not RESUME_REASON.strip(): raise ValueError('Resume needs a recorded reason.')
    if RECOVER_STALE_CLAIM and not RESUME: raise ValueError('Stale claim recovery requires resume.')
    if not RESUME and OUT.exists(): raise FileExistsError(f'Run output exists; inspect it: {OUT}')
    # Check the real environment again immediately before requesting sealed access.
    ready = run_command([ROOT/'scripts/r5_n1_sealed.py', '--root', ROOT, 'plan', '--shards', SHARDS], capture=True)
    if 'Locks, bundles and split files verified: YES' not in ready or 'Ready to run on this machine: YES' not in ready:
        raise RuntimeError('Preflight no longer passes.')
    args = [ROOT/'scripts/r5_n1_sealed.py', '--root', ROOT, 'run', '--shards', SHARDS,
            '--confirm-sealed-run', '--out', OUT]
    if RESUME: args += ['--resume', '--resume-reason', RESUME_REASON]
    if RECOVER_STALE_CLAIM: args.append('--recover-stale-claim')
    run_command(args)
    summary = json.loads((OUT/'integrity_report.json').read_text(encoding='utf-8'))
    if summary.get('status') != 'complete' or summary.get('passed') is not True:
        raise RuntimeError('Sealed integrity did not pass; inspect the persistent evidence.')
    print('Clean sealed scoring completed. Evidence:', OUT)


## Analysis ? disabled by default

After a successfully completed clean run, set `RUN_ANALYSIS=True` and rerun this cell. It verifies the saved predictions and produces metrics, paired speaker bootstrap intervals and plots without rereading test audio or split rows. Existing analysis output is never overwritten. The whole run folder is already on Drive; copy it locally for review.


In [ ]:
if not RUN_ANALYSIS:
    print('Analysis disabled.')
else:
    if not TESTS_OK or not PREFLIGHT_OK or source_snapshot() != PREFLIGHT_SNAPSHOT:
        raise RuntimeError('Rerun tests and preflight with the current code before analysis.')
    if not (OUT/'integrity_report.json').is_file(): raise FileNotFoundError('No completed run evidence found.')
    run_command([ROOT/'scripts/r5_n1_clean_analysis.py', '--root', ROOT, '--run', OUT])
    print('Analysis:', OUT/'analysis_clean_v1')
